# Fast Custom CNN + Live Product Detection (17 categories)

This notebook trains a **MobileNetV2 transfer-learning CNN classifier** from your own product photos and uses **YOLO-World** to propose real object boxes in the live webcam. The CNN labels each detected crop; boxes update as the camera view changes. No Supabase is used.

**Important:** 5 images per category is only a tiny starter dataset. Augmentation helps but cannot create new real-world variety. The notebook does not claim guaranteed accuracy. For stronger detection of every category, annotate and train a custom YOLO detector with bounding boxes. YOLO-World may miss unusual objects such as wires, erasers, or toothpaste; the CNN can only classify objects that have been cropped/detected.

Run cells from top to bottom. Select a GPU in **Runtime → Change runtime type → T4 GPU** if available.

In [1]:
!pip -q install tensorflow==2.16.1 ultralytics gradio pillow scikit-learn ddgs requests


ERROR: Could not find a version that satisfies the requirement tensorflow==2.16.1 (from versions: 2.20.0rc0, 2.20.0, 2.21.0rc0, 2.21.0rc1, 2.21.0, 2.22.0rc0)
ERROR: No matching distribution found for tensorflow==2.16.1


## 1. Automatically build a starter dataset from online images

You do **not** need to upload a dataset. The next cell searches public image-search results for each category, downloads usable images, and stores them in class folders automatically. Search results can include incorrect or copyrighted images; this is a quick prototype dataset, not a curated benchmark. Review/replace poor images if you want better accuracy.

The notebook attempts to download 20 images per category. Search engines can rate-limit or block requests, so some categories may end up with fewer images.


In [3]:
%pip install -q -U ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.2 MB/s eta 0:00:00


In [4]:
import os, time, requests, pathlib, random, io
from PIL import Image, ImageFile
from ddgs import DDGS
ImageFile.LOAD_TRUNCATED_IMAGES = True

CLASSES = ['biscuit','chips','pen','scale','phone','calculator','headphone','laptop','books','pencils','erasers','keyboard','mouse','toothpaste','ball','toothbrush','wires']
SEARCH_TERMS = {
 'biscuit':'biscuit packet packaged biscuits product photo',
 'chips':'potato chips packet snack packet product photo',
 'pen':'ballpoint pen stationery product photo',
 'scale':'ruler measuring scale stationery product photo',
 'phone':'smartphone mobile phone product photo',
 'calculator':'handheld calculator product photo',
 'headphone':'headphones headset product photo',
 'laptop':'laptop computer product photo',
 'books':'school books notebooks stack product photo',
 'pencils':'pencils stationery product photo',
 'erasers':'rubber eraser stationery product photo',
 'keyboard':'computer keyboard product photo',
 'mouse':'computer mouse product photo',
 'toothpaste':'toothpaste tube product photo',
 'ball':'sports ball product photo',
 'toothbrush':'toothbrush product photo',
 'wires':'electrical wires cables product photo'
}
DATA_DIR='/content/product_images'
os.makedirs(DATA_DIR,exist_ok=True)
TARGET_PER_CLASS=20  # increase to 30-50 later for a better starter dataset
session=requests.Session()
session.headers.update({'User-Agent':'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/124 Safari/537.36'})
def download_class(class_name, query, target=TARGET_PER_CLASS):
    folder=os.path.join(DATA_DIR,class_name); os.makedirs(folder,exist_ok=True)
    existing=list(pathlib.Path(folder).glob('*.jpg'))
    if len(existing)>=target: return len(existing)
    try:
        results=DDGS().images(query, region='us-en', safesearch='moderate', max_results=target*3)
    except Exception as e:
        print(f'Image search failed for {class_name}: {e}'); return len(existing)
    saved=len(existing); seen=set()
    for item in results:
        if saved>=target: break
        url=item.get('image') or item.get('thumbnail')
        if not url or url in seen: continue
        seen.add(url)
        try:
            response=session.get(url,timeout=10); response.raise_for_status()
            im=Image.open(io.BytesIO(response.content)).convert('RGB')
            if im.width<80 or im.height<80: continue
            im.thumbnail((900,900))
            out=os.path.join(folder,f'{class_name}_{saved+1:03d}.jpg')
            im.save(out,'JPEG',quality=90)
            saved+=1
        except Exception:
            continue
    return saved

counts={}
for cls in CLASSES:
    counts[cls]=download_class(cls,SEARCH_TERMS[cls])
    print(f'{cls:12s}: {counts[cls]} images downloaded')
    time.sleep(1)  # reduce search throttling

print('\nDataset folder:',DATA_DIR)
missing=[c for c,n in counts.items() if n<2]
if missing:
    raise RuntimeError('Too few images for: '+', '.join(missing)+'. Rerun this cell later or change the search term.')
print('Total images:',sum(counts.values()))


biscuit     : 20 images downloaded
chips       : 20 images downloaded
pen         : 20 images downloaded
scale       : 20 images downloaded


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


phone       : 20 images downloaded
calculator  : 20 images downloaded
headphone   : 20 images downloaded
laptop      : 20 images downloaded
books       : 20 images downloaded
pencils     : 20 images downloaded
erasers     : 20 images downloaded
keyboard    : 20 images downloaded
mouse       : 20 images downloaded
toothpaste  : 20 images downloaded
ball        : 20 images downloaded
toothbrush  : 20 images downloaded
wires       : 20 images downloaded

Dataset folder: /content/product_images
Total images: 340


In [5]:
import tensorflow as tf, numpy as np, pathlib, json, time
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
CLASSES = ['biscuit','chips','pen','scale','phone','calculator','headphone','laptop','books','pencils','erasers','keyboard','mouse','toothpaste','ball','toothbrush','wires']
IMG_SIZE=(160,160); BATCH=16; SEED=42
# Validate folders and collect images.
valid_ext={'.jpg','.jpeg','.png','.webp','.bmp'}
paths=[]; labels=[]
for idx, name in enumerate(CLASSES):
    folder=os.path.join(DATA_DIR,name)
    if not os.path.isdir(folder): print('MISSING FOLDER:', name); continue
    files_in=[p for p in pathlib.Path(folder).rglob('*') if p.suffix.lower() in valid_ext]
    print(f'{name:12s}: {len(files_in)} images')
    for p in files_in: paths.append(str(p)); labels.append(idx)
assert len(set(labels)) >= 2, 'Need images in at least two class folders.'
assert all(os.path.isdir(os.path.join(DATA_DIR,c)) for c in CLASSES), 'Please add all 17 folders before training.'
from collections import Counter
print('Total images:',len(paths),' | smallest class:',min(Counter(labels).values()))

biscuit     : 20 images
chips       : 20 images
pen         : 20 images
scale       : 20 images
phone       : 20 images
calculator  : 20 images
headphone   : 20 images
laptop      : 20 images
books       : 20 images
pencils     : 20 images
erasers     : 20 images
keyboard    : 20 images
mouse       : 20 images
toothpaste  : 20 images
ball        : 20 images
toothbrush  : 20 images
wires       : 20 images
Total images: 340  | smallest class: 20


## 2. Train the fast CNN

MobileNetV2 is pretrained on ImageNet; we reuse its visual features and train a small 17-class head. This is much faster than training a CNN from scratch. With only five images per class, the displayed validation score is a rough sanity check, **not** a reliable real-world accuracy estimate.

In [6]:
# Stratified split only when each class has at least 2 images; keep a small validation set.
X_train, X_val, y_train, y_val = train_test_split(np.array(paths), np.array(labels), test_size=0.2, random_state=SEED, stratify=np.array(labels))
AUTOTUNE=tf.data.AUTOTUNE
def load_image(path, label):
    raw=tf.io.read_file(path)
    img=tf.image.decode_image(raw, channels=3, expand_animations=False)
    img=tf.image.resize(img, IMG_SIZE)
    img=preprocess_input(tf.cast(img,tf.float32))
    return img, tf.one_hot(label, len(CLASSES))
def make_ds(x,y,training=False):
    ds=tf.data.Dataset.from_tensor_slices((x,y))
    if training: ds=ds.shuffle(len(x),seed=SEED)
    ds=ds.map(load_image,num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)
    return ds
train_ds=make_ds(X_train,y_train,True); val_ds=make_ds(X_val,y_val)
augment=tf.keras.Sequential([layers.RandomFlip('horizontal'),layers.RandomRotation(0.08),layers.RandomZoom(0.12),layers.RandomContrast(0.12)],name='augmentation')
base=MobileNetV2(input_shape=(*IMG_SIZE,3),include_top=False,weights='imagenet',pooling='avg')
base.trainable=False
inputs=layers.Input(shape=(*IMG_SIZE,3))
x=augment(inputs)
x=base(x,training=False)
x=layers.Dropout(0.25)(x)
outputs=layers.Dense(len(CLASSES),activation='softmax')(x)
cnn=models.Model(inputs,outputs)
cnn.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss='categorical_crossentropy',metrics=['accuracy'])
history=cnn.fit(train_ds,validation_data=val_ds,epochs=8,verbose=1)
val_probs=cnn.predict(val_ds,verbose=0)
val_pred=val_probs.argmax(axis=1)
print('Validation accuracy (small split; not a reliable generalization estimate):', round(accuracy_score(y_val,val_pred),3))
print(classification_report(y_val,val_pred,labels=list(range(len(CLASSES))),target_names=CLASSES,zero_division=0))
cnn.save('/content/product_cnn_mobilenetv2.keras')
with open('/content/product_classes.json','w') as f: json.dump(CLASSES,f)
print('Saved CNN and class list.')

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 12s 142ms/step - accuracy: 0.1250 - loss: 3.1912 - val_accuracy: 0.2353 - val_loss: 2.3306
Epoch 2/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.4118 - loss: 1.8964 - val_accuracy: 0.4853 - val_loss: 1.5900
Epoch 3/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step - accuracy: 0.6765 - loss: 1.1494 - val_accuracy: 0.6176 - val_loss: 1.2068
Epoch 4/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 0.7500 - loss: 0.8360 - val_accuracy: 0.7059 - val_loss: 1.0366
Epoch 5/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.8493 - loss: 0.6341 - val_accuracy: 0.7500 - val_loss: 0.9081
Epoch 6/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 62ms/step - accuracy: 0.8860 - loss: 0.5161 - val_accuracy: 0.7647 - val_loss: 0.8326
Epoch 7/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 57ms/step - accuracy: 0.8971 - loss: 0.4104 - val_accuracy: 0.7794 - val_loss: 0.7835
Epoch 8/8
17/17 ━━━━━━━━━━━━━━━━━━━━ 1s 58ms/step - accuracy: 0.904

## 3. Load the live detector

YOLO-World proposes real bounding boxes; the trained CNN classifies each detected crop into your 17 custom categories. Inference is run on resized frames for speed. On a free CPU runtime it may not feel instant; GPU is recommended.

In [8]:
%pip install -q -U ultralytics gradio opencv-python-headless

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.2/97.2 kB 9.8 MB/s eta 0:00:00


In [9]:
from ultralytics import YOLO
import cv2, gradio as gr, PIL.Image as Image
detector=YOLO('yolov8s-worldv2.pt')
PROMPTS=['biscuit packet','chips packet','pen','ruler','mobile phone','calculator','headphones','laptop','book','pencils','eraser','computer keyboard','computer mouse','toothpaste tube','ball','toothbrush','wires or cables']
detector.set_classes(PROMPTS)
cnn=tf.keras.models.load_model('/content/product_cnn_mobilenetv2.keras')
print('Detector and CNN ready. Webcam inference will start in the next cell.')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 37 packages in 768ms
Prepared 2 packages in 2.11s
Installed 2 packages in 1ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@7ffa84b3bfa40c42ecc1c77147a855e69cb2dd40)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 3.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|████████████████████████████████████████| 338M/338M [00:03<00:00, 111MiB/s]


Detector and CNN ready. Webcam inference will start in the next cell.


In [5]:

from ultralytics import YOLO
import tensorflow as tf
import numpy as np
import cv2
import gradio as gr
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# Load the object detector
detector = YOLO("yolov8s-worldv2.pt")

PROMPTS = [
    "biscuit packet", "potato chips packet", "ballpoint pen",
    "ruler measuring scale", "smartphone", "handheld calculator",
    "headphones headset", "laptop computer", "school book",
    "pencils", "eraser rubber", "computer keyboard",
    "computer mouse", "toothpaste tube", "sports ball",
    "toothbrush", "electrical wire cable"
]

detector.set_classes(PROMPTS)

# Reuse your previously trained TensorFlow CNN
cnn = tf.keras.models.load_model(
    "/content/product_cnn_mobilenetv2.keras"
)

print("YOLO detector and TensorFlow CNN loaded successfully!")


Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/gradio/queueing.py", line 848, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 2698, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/gradio/blocks.py", line 1963, in call_function
    prediction = await anyio.to_thread.run_sync(  # type: ignore
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        fn, *processed_input, limiter=self.limiter
        ^^^^^^^^^^^^^^

YOLO detector and TensorFlow CNN loaded successfully!


In [2]:
%pip install -q gradio
import gradio as gr
import cv2
import numpy as np

In [6]:

import gradio as gr
import cv2
import numpy as np
import torch
import time

# Use GPU automatically when available
DEVICE = 0 if torch.cuda.is_available() else "cpu"
USE_HALF = torch.cuda.is_available()

# Lower input size = faster inference
DETECT_SIZE = 512
MAX_DETECTIONS = 15
DETECT_CONF = 0.08

def recognize_frame(frame):
    if frame is None:
        return None

    frame = np.asarray(frame, dtype=np.uint8)
    h, w = frame.shape[:2]

    # Resize once, keeping the original aspect ratio
    scale = min(DETECT_SIZE / max(h, w), 1.0)
    sw, sh = max(1, int(w * scale)), max(1, int(h * scale))
    small = cv2.resize(frame, (sw, sh), interpolation=cv2.INTER_AREA) if scale < 1 else frame

    # Fast real-time object detection
    result = detector.predict(
        source=small,
        imgsz=DETECT_SIZE,
        conf=DETECT_CONF,
        iou=0.50,
        max_det=MAX_DETECTIONS,
        device=DEVICE,
        half=USE_HALF,
        verbose=False
    )[0]

    output = frame.copy()

    if result.boxes is None or len(result.boxes) == 0:
        cv2.putText(
            output, "No object found - move closer / improve lighting",
            (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 165, 255), 2
        )
        return output

    # Convert detected coordinates back to original frame coordinates
    sx, sy = w / sw, h / sh
    boxes, crops = [], []

    for box in result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        x1, y1, x2, y2 = int(x1*sx), int(y1*sy), int(x2*sx), int(y2*sy)

        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        if x2 - x1 < 12 or y2 - y1 < 12:
            continue

        detector_id = int(box.cls[0].item())
        detector_conf = float(box.conf[0].item())

        crop = cv2.resize(
            frame[y1:y2, x1:x2], IMG_SIZE,
            interpolation=cv2.INTER_AREA
        ).astype(np.float32)

        crops.append(preprocess_input(crop))
        boxes.append((x1, y1, x2, y2, detector_id, detector_conf))

    # Classify every crop in one CNN batch instead of one call per object
    if crops:
        predictions = cnn.predict(
            np.stack(crops), batch_size=MAX_DETECTIONS, verbose=0
        )

        for (x1, y1, x2, y2, det_id, det_conf), probs in zip(boxes, predictions):
            cnn_id = int(np.argmax(probs))
            cnn_conf = float(probs[cnn_id])

            # Prefer the CNN when confident; otherwise use detector's class
            if cnn_conf >= 0.40:
                label = CLASSES[cnn_id]
                confidence = cnn_conf
            elif 0 <= det_id < len(CLASSES):
                label = CLASSES[det_id]
                confidence = det_conf
            else:
                label = "product?"
                confidence = max(cnn_conf, det_conf)

            text = f"{label} {confidence:.0%}"

            # Green bounding box and label
            cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
            (tw, th), _ = cv2.getTextSize(
                text, cv2.FONT_HERSHEY_SIMPLEX, 0.58, 2
            )
            top = max(0, y1 - th - 8)
            cv2.rectangle(
                output, (x1, top), (min(w, x1 + tw + 8), top + th + 8),
                (0, 255, 0), -1
            )
            cv2.putText(
                output, text, (x1 + 4, top + th + 2),
                cv2.FONT_HERSHEY_SIMPLEX, 0.58, (0, 0, 0), 2
            )

    return output


demo = gr.Interface(
    fn=recognize_frame,
    inputs=gr.Image(
        sources=["webcam"],
        streaming=True,
        type="numpy",
        label="Live webcam"
    ),
    outputs=gr.Image(type="numpy", label="Detected products"),
    live=True,
    stream_every=0.15,
    title="Fast Live Product Detection",
    description="Show products close to the camera with bright lighting."
)

demo.launch(share=True, debug=False)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://575583d9fff667f289.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 4. Export trained files

Download the trained CNN and class list if you want to reuse them. The YOLO-World weights are downloaded automatically by Ultralytics when first used.


In [ ]:
from google.colab import files
files.download('/content/product_cnn_mobilenetv2.keras')
files.download('/content/product_classes.json')